In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve,average_precision_score)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("C:\\Users\\delta\\Predictive-maintenence-iot\\dataset\\ai4i2020.csv")
y = df["Machine failure"]
X = df.drop(columns=["UDI","Product ID","Machine failure"])
X = pd.get_dummies(X,columns=["Type"],drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("model", LogisticRegression(max_iter=1000))])
pipeline.fit(X_train, y_train)
failure_probability = pipeline.predict_proba(X_test)[:,1]
precision, recall, thresholds = precision_recall_curve(y_test,failure_probability)
average_precision = average_precision_score(y_test,failure_probability)
f1_scores = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)  # Adding a small value to avoid division by zero
threshold_results = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": precision[:-1],
    "Recall": recall[:-1],
    "F1 Score": f1_scores})
best_recall = threshold_results.loc[threshold_results["Recall"].idxmax()]
best_precision = threshold_results.loc[threshold_results["Precision"].idxmax()]
best_balance = threshold_results.loc[threshold_results["F1 Score"].idxmax()]
comparison = pd.DataFrame({
    "Strategy":["Maximum Recall","Maximum Precision","Balanced Performance"],
    "Threshold":[best_recall["Threshold"],best_precision["Threshold"],best_balance["Threshold"]],
    "Precision":[best_recall["Precision"],best_precision["Precision"],best_balance["Precision"]],
    "Recall":[best_recall["Recall"],best_precision["Recall"],best_balance["Recall"]],
    "F1 Score":[best_recall["F1 Score"],best_precision["F1 Score"],best_balance["F1 Score"]]})
results = pd.DataFrame({
    "Accuracy":[pipeline.score(X_test, y_test)],
    "Precision":[best_balance["Precision"]],
    "Recall":[best_balance["Recall"]],
    "F1 Score":[best_balance["F1 Score"]]})

In [ ]:
final_metrics = pd.DataFrame({
    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Average Precision"
    ],
    "Value":[
        results["Accuracy"].mean(),
        results["Precision"].mean(),
        results["Recall"].mean(),
        results["F1 Score"].mean(),
        average_precision
    ]
})
final_metrics

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(final_metrics["Metric"],final_metrics["Value"])
plt.ylim(0,1)
plt.title("Overall Model Performance")
plt.ylabel("Score")
plt.grid(axis="y")
plt.savefig("overall_model_performance.png")
plt.show()

In [ ]:
comparison

In [ ]:
comparison.set_index("Strategy")[["Precision","Recall","F1 Score"]].plot(kind="bar",figsize=(10,6))
plt.title("Threshold Strategy Comparison")
plt.ylabel("Score")
plt.grid(True)
plt.savefig("Overall_threshold_strategies_comparison.png")
plt.show()

In [ ]:
best_threshold = pd.DataFrame({
    "Parameter":[
        "Selected Threshold",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value":[
        best_balance["Threshold"],
        best_balance["Precision"],
        best_balance["Recall"],
        best_balance["F1 Score"]
    ]
})
best_threshold

In [ ]:
threshold_results.describe()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(threshold_results["Precision"],bins=20,edgecolor="black")
plt.title("Precision Distribution")
plt.xlabel("Precision")
plt.ylabel("Frequency")
plt.savefig("Precision_Distribution.png")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(threshold_results["Recall"],bins=20,edgecolor="black")
plt.title("Recall Distribution")
plt.xlabel("Recall")
plt.ylabel("Frequency")
plt.savefig("recall_distribution.png")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(threshold_results["F1 Score"],bins=20,edgecolor="black")
plt.title("F1 Score Distribution")
plt.xlabel("F1 Score")
plt.ylabel("Frequency")
plt.savefig("f1_score_distribution.png")
plt.show()

In [ ]:
top10 = threshold_results.sort_values(by="F1 Score",ascending=False)
top10.head(10)

In [ ]:
ranking = final_metrics.sort_values(by="Value",ascending=False)
ranking

In [ ]:
strengths = pd.DataFrame({
    "Strength":[
        "Balanced Dataset",
        "Reduced Class Bias",
        "Reliable Recall",
        "Stable Precision",
        "Good Threshold Selection"
    ],
    "Observation":[
        "SMOTE balanced minority class",
        "Improved learning for failures",
        "Failure detection improved",
        "False alarms reduced",
        "Best F1 threshold selected"
    ]
})
strengths

In [ ]:
limitations = pd.DataFrame({
    "Limitation":[
        "Synthetic samples",
        "Threshold sensitivity",
        "Rare failure events",
        "Industrial variability"
    ],
    "Possible Improvement":[
        "Collect more real failures",
        "Dynamic threshold tuning",
        "More operational data",
        "Test additional ML models"
    ]
})
limitations

In [ ]:
findings = pd.DataFrame({
    "Finding":[
        "Class imbalance successfully handled",
        "Prediction probabilities improved evaluation",
        "Precision-Recall analysis supported threshold selection",
        "Balanced threshold achieved best trade-off",
        "Model suitable for predictive maintenance applications"
    ]
})
findings

In [ ]:
conclusion = """
FINAL CONCLUSION

The predictive maintenance model demonstrated effective performance after
handling class imbalance using SMOTE.

Precision-Recall analysis provided a more informative evaluation than
accuracy alone because machine failure is a minority class.

Threshold analysis identified an operating point that balances failure
detection and false alarm reduction.

The final model can support maintenance decision-making by identifying
machines with a higher probability of failure while maintaining a
balanced trade-off between Precision and Recall.

Future work may include evaluating advanced machine learning algorithms,
feature engineering, and real-time IoT deployment.
"""
print(conclusion)